## **🎨 Real-Time Multi-Object Colorization — Task 1**



---
## 📦 Section 1 — Installation

In [ ]:
# ================================================================
# SECTION 1 — INSTALLATION
# All packages pinned. ffmpeg is pre-installed on Colab.
# ================================================================
import subprocess, sys

PKGS = [
    "ultralytics>=8.2.0",
    "gradio>=4.20.0",
    "opencv-python-headless>=4.9.0",
    "Pillow>=10.0.0",
    "tqdm>=4.66.0",
    "torchvision>=0.17.0",
    "transformers>=4.38.0",
    "timm>=0.9.0",
    "numpy>=1.24.0",
]
for p in PKGS:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p])

# Verify ffmpeg (pre-installed on Colab, needed for video encoding)
r = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
if r.returncode == 0:
    ver = r.stdout.split("\n")[0]
    print(f"✅ ffmpeg: {ver[:60]}")
else:
    print("⚠️  ffmpeg not found — installing...")
    subprocess.check_call(["apt-get", "install", "-y", "-q", "ffmpeg"])

print("\n✅ All packages ready.")


✅ ffmpeg: ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-202

✅ All packages ready.


---
## ⚙️ Section 2 — Imports & Device

In [ ]:
# ================================================================
# SECTION 2 — IMPORTS & DEVICE CONFIGURATION
# ================================================================
import os, gc, time, warnings, tempfile, traceback, shutil, threading
from pathlib import Path
from typing import Dict, Tuple, Optional, List, Generator

import cv2
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm import tqdm
import gradio as gr
from ultralytics import YOLO
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation

warnings.filterwarnings("ignore")

DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_FP16 = DEVICE.type == "cuda"

print(f"Device  : {DEVICE}")
print(f"FP16    : {USE_FP16}")
if DEVICE.type == "cuda":
    p = torch.cuda.get_device_properties(0)
    print(f"GPU     : {p.name}  VRAM {p.total_memory/1e9:.1f} GB")
print(f"PyTorch : {torch.__version__}")


Device  : cuda
FP16    : True
GPU     : Tesla T4  VRAM 15.6 GB
PyTorch : 2.11.0+cu128


---
## 🎨 Section 3 — ADE20K Color Palette (150 Classes)

In [ ]:
# ================================================================
# SECTION 3 — ADE20K 150-CLASS COLOR PALETTE
# BGR format. None = preserve original pixels (used for person).
# All required project classes are explicitly mapped.
# ================================================================

ADE20K_COLORS: Dict[int, Optional[Tuple]] = {
    0:  (140, 160, 170),  # wall
    1:  (140, 180, 210),  # building      → sandy brown  #D2B48C
    2:  (235, 206, 135),  # sky           → sky blue     #87CEEB
    3:  (90,  90,  90),   # floor
    4:  (34,  100, 34),   # tree          → forest green #228B22
    5:  (180, 180, 180),  # ceiling
    6:  (128, 128, 128),  # road          → gray         #808080
    7:  (0,   200, 100),  # grass         → lawn green
    8:  (160, 160, 130),  # sidewalk
    9:  None,             # person        → PRESERVE PIXELS
    10: (50,  80,  130),  # earth/ground
    11: (100, 80,  60),   # door
    12: (80,  100, 120),  # table
    13: (100, 120, 90),   # mountain
    14: (20,  160, 80),   # plant
    15: (220, 180, 200),  # curtain
    16: (100, 80,  120),  # chair
    17: (68,  68,  255),  # car           → red          #FF4444
    18: (255, 144, 30),   # water         → dodger blue  #1E90FF
    21: (255, 144, 30),   # sea
    23: (70,  90,  70),   # field/lawn
    26: (140, 180, 210),  # house
    29: (255, 144, 30),   # river
    84: (0,   165, 255),  # bus
    85: (128, 0,   128),  # bicycle
    86: (128, 0,   128),  # motorbike
    103:(255, 144, 30),   # lake
}

def _build_palette() -> np.ndarray:
    pal = np.zeros((256, 3), dtype=np.uint8)
    for i in range(256):
        h=(i*137.508)%360; s,v=0.65,0.80; c=v*s
        x=c*(1-abs((h/60)%2-1)); m=v-c
        if h<60:    r,g,b=c,x,0
        elif h<120: r,g,b=x,c,0
        elif h<180: r,g,b=0,c,x
        elif h<240: r,g,b=0,x,c
        elif h<300: r,g,b=x,0,c
        else:       r,g,b=c,0,x
        pal[i]=[int((b+m)*255),int((g+m)*255),int((r+m)*255)]
    for idx,bgr in ADE20K_COLORS.items():
        if bgr is not None and idx<256: pal[idx]=bgr
    return pal

ADE20K_PALETTE = _build_palette()

YOLO_COLORS: Dict[str, Optional[Tuple]] = {
    "person":None, "bicycle":(128,0,128), "car":(68,68,255),
    "motorcycle":(128,0,128), "airplane":(211,211,211),
    "bus":(0,165,255), "train":(60,20,220), "truck":(0,140,255),
    "boat":(139,90,43), "traffic light":(0,255,255),
    "stop sign":(0,0,200), "cat":(203,192,255), "dog":(230,216,173),
    "horse":(19,69,139), "bird":(255,191,0), "chair":(180,105,255),
    "bench":(42,42,165),
}
def get_yolo_color(name:str)->Optional[Tuple]:
    n=name.lower().strip()
    if n in YOLO_COLORS: return YOLO_COLORS[n]
    h=hash(n)&0xFFFFFF
    return (h&0xFF,(h>>8)&0xFF,(h>>16)&0xFF)

print("✅ ADE20K palette ready — sky/road/building/grass/tree/water/car all mapped")


✅ ADE20K palette ready — sky/road/building/grass/tree/water/car all mapped


---
## 🧠 Section 4 — Model Loading

In [ ]:
# ================================================================
# SECTION 4 — MODEL LOADING
# SegFormer-B2 (ADE20K 150 cls) + YOLOv8s-seg (COCO 80 cls)
# Both in FP16 on GPU.
# ================================================================

print("Loading SegFormer-B2...")
SF_PROC  = SegformerImageProcessor.from_pretrained(
               "nvidia/segformer-b2-finetuned-ade-512-512")
SF_MODEL = SegformerForSemanticSegmentation.from_pretrained(
               "nvidia/segformer-b2-finetuned-ade-512-512"
           ).to(DEVICE).eval()
if USE_FP16: SF_MODEL = SF_MODEL.half()
print(f"  ✅ SegFormer-B2 [{DEVICE}, {'FP16' if USE_FP16 else 'FP32'}]")

print("Loading YOLOv8s-seg...")
YOLO_MODEL = YOLO("yolov8s-seg.pt")
YOLO_MODEL.to(DEVICE)
print(f"  ✅ YOLOv8s-seg  [{DEVICE}]")


@torch.inference_mode()
def segment_frame(bgr: np.ndarray, inf_size: int = 512) -> np.ndarray:
    """SegFormer-B2 → (H,W) uint8 ADE20K class IDs."""
    H, W = bgr.shape[:2]
    pil = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    sc  = inf_size / max(H, W)
    if sc < 1.0:
        pil = pil.resize((max(1,int(W*sc)), max(1,int(H*sc))), Image.BILINEAR)
    inp = SF_PROC(images=pil, return_tensors="pt")
    pv  = inp["pixel_values"].to(DEVICE)
    if USE_FP16: pv = pv.half()
    logits = SF_MODEL(pixel_values=pv).logits          # (1,150,H',W')
    logits_up = F.interpolate(logits.float(), (H,W),
                              mode="bilinear", align_corners=False)
    mask = logits_up.squeeze(0).argmax(0).cpu().numpy().astype(np.uint8)
    # Free GPU tensors immediately
    del logits, logits_up, pv
    return mask

print("\n✅ Models ready.")


Loading SegFormer-B2...


Loading weights: 100%|██████████| 380/380 [00:00<00:00, 2851.52steps/s]


  ✅ SegFormer-B2 [cuda, FP16]
Loading YOLOv8s-seg...
  ✅ YOLOv8s-seg  [cuda]

✅ Models ready.


---
## 🖌️ Section 5 — Colorization Engine

In [ ]:
# ================================================================
# SECTION 5 — COLORIZATION ENGINE
# Key memory fix: retina_masks=False + explicit tensor deletion.
# ================================================================

class TemporalSmoother:
    def __init__(self, alpha:float=0.65):
        self.alpha=alpha; self._prev=None
    def smooth(self, cur:np.ndarray)->np.ndarray:
        if self._prev is None or self._prev.shape!=cur.shape:
            self._prev=cur.astype(np.float32); return cur
        s=self.alpha*cur.astype(np.float32)+(1-self.alpha)*self._prev
        self._prev=s.copy(); return s.astype(np.uint8)
    def reset(self): self._prev=None

SMOOTHER = TemporalSmoother(0.45)


def colorize_frame(bgr:np.ndarray, seg_mask:np.ndarray, yolo_results,
                   alpha:float=0.30, preserve_persons:bool=True,
                   draw_boxes:bool=True, draw_labels:bool=True,
                   temporal_smooth:bool=True) -> np.ndarray:
    """
    Colorize one frame.  All YOLO tensors are moved to CPU and deleted
    before returning to prevent GPU memory accumulation.
    """
    H, W = bgr.shape[:2]

    # 1. Palette lookup
    safe = np.clip(seg_mask, 0, 255)
    color_layer = ADE20K_PALETTE[safe]

    # 2. Temporal smoothing
    if temporal_smooth:
        color_layer = SMOOTHER.smooth(color_layer)

    # 3. Alpha blend
    colorized = cv2.addWeighted(
        bgr.astype(np.float32),         1.0 - alpha,
        color_layer.astype(np.float32), alpha, 0.0
    ).astype(np.uint8)

    # 4. Restore person pixels (ADE20K class 9)
    if preserve_persons:
        pm = (seg_mask == 9)
        if pm.any(): colorized[pm] = bgr[pm]

    # 5. YOLO overlay
    if yolo_results is not None:
        for result in yolo_results:
            if result.boxes is None: continue

            # Pull everything to CPU immediately and delete GPU tensors
            boxes_np   = result.boxes.xyxy.cpu().numpy().astype(int)
            confs_np   = result.boxes.conf.cpu().numpy()
            cls_np     = result.boxes.cls.cpu().numpy().astype(int)
            names      = result.names

            # masks: use cpu-side data only; retina_masks=False so small
            has_masks  = result.masks is not None
            masks_np   = result.masks.data.cpu().numpy() if has_masks else None

            for i,(box,conf,cid) in enumerate(zip(boxes_np,confs_np,cls_np)):
                x1,y1,x2,y2 = box
                x1,y1 = max(0,x1), max(0,y1)
                x2,y2 = min(W,x2), min(H,y2)
                cn     = names.get(cid, str(cid))
                col    = get_yolo_color(cn)

                # Instance mask fill (only for non-person)
                if col is not None and has_masks and i < len(masks_np):
                    inst = cv2.resize(masks_np[i], (W,H),
                                      interpolation=cv2.INTER_NEAREST)
                    bm = (inst > 0.5)
                    if bm.any():
                        fill = np.full((H,W,3), col, np.float32)
                        blend = (0.45*fill+0.55*bgr.astype(np.float32)).astype(np.uint8)
                        colorized[bm] = blend[bm]

                # Restore person pixels (YOLO detection)
                if preserve_persons and cn.lower() == "person":
                    if has_masks and i < len(masks_np):
                        inst = cv2.resize(masks_np[i], (W,H),
                                          interpolation=cv2.INTER_NEAREST)
                        pm2 = (inst > 0.5)
                        if pm2.any(): colorized[pm2] = bgr[pm2]
                    else:
                        colorized[y1:y2,x1:x2] = bgr[y1:y2,x1:x2]

                # Bounding box
                if draw_boxes and col is not None:
                    cv2.rectangle(colorized,(x1,y1),(x2,y2),col,2,cv2.LINE_AA)

                # Label
                if draw_labels and col is not None:
                    lbl=f"{cn} {conf:.0%}"
                    font,fs,th=cv2.FONT_HERSHEY_SIMPLEX,0.50,1
                    (tw,lh),bl=cv2.getTextSize(lbl,font,fs,th)
                    ty=max(y1-4,lh+4)
                    cv2.rectangle(colorized,(x1,ty-lh-bl),(x1+tw+4,ty+bl),col,cv2.FILLED)
                    bri=0.299*col[2]+0.587*col[1]+0.114*col[0]
                    tc=(0,0,0) if bri>128 else (255,255,255)
                    cv2.putText(colorized,lbl,(x1+2,ty),font,fs,tc,th,cv2.LINE_AA)

    return colorized


def seg_vis(bgr:np.ndarray, mask:np.ndarray, alpha:float=0.70)->np.ndarray:
    safe=np.clip(mask,0,255)
    cl=ADE20K_PALETTE[safe]
    return cv2.addWeighted(bgr.astype(np.float32),1-alpha,
                           cl.astype(np.float32),alpha,0.0).astype(np.uint8)

print("✅ Colorization engine ready.")


✅ Colorization engine ready.


---
## 🎬 Section 6 — ffmpeg Video Encoder (Replaces cv2.VideoWriter)

**This is the fix for the 99% hang.**  
`cv2.VideoWriter.release()` blocks for 30–60 s finalising codec buffers on Colab.  
We write raw frames to a pipe and let `ffmpeg` handle encoding in a subprocess — it never blocks the Python thread.

In [ ]:
# ================================================================
# SECTION 6 — FFMPEG-BASED VIDEO ENCODER
#
# Why ffmpeg instead of cv2.VideoWriter?
#   cv2.VideoWriter.release() can HANG for 30-60 seconds on Colab
#   while it finalises the codec container. This is the root cause
#   of the "stuck at 99% → broken connection" error.
#
# ffmpeg writes frames from a stdin pipe and exits cleanly when
# the pipe is closed — no hanging, no codec buffer stall.
# ================================================================
import subprocess as sp

class FfmpegWriter:
    """
    Writes BGR numpy frames to an MP4 file via ffmpeg stdin pipe.
    Produces H.264 video that plays natively in browsers/Gradio.
    No hanging on close() — ffmpeg exits when stdin is closed.
    """

    def __init__(self, output_path: str, fps: float, width: int, height: int):
        self.path   = output_path
        self.fps    = fps
        self.width  = width
        self.height = height
        self._proc  = None
        self._open()

    def _open(self):
        cmd = [
            "ffmpeg",
            "-y",                          # overwrite output
            "-f", "rawvideo",              # input format
            "-vcodec", "rawvideo",
            "-s", f"{self.width}x{self.height}",
            "-pix_fmt", "bgr24",           # OpenCV default is BGR
            "-r", str(self.fps),           # input framerate
            "-i", "pipe:0",                # read from stdin
            "-vcodec", "libx264",          # H.264 — plays in Gradio
            "-preset", "ultrafast",        # fastest encoding
            "-crf", "23",                  # quality (lower = better)
            "-pix_fmt", "yuv420p",         # browser-compatible pixel fmt
            "-movflags", "+faststart",     # web-optimised MP4 header
            "-loglevel", "error",          # suppress ffmpeg noise
            self.path,
        ]
        self._proc = sp.Popen(cmd, stdin=sp.PIPE)

    def write(self, frame: np.ndarray):
        """Write one BGR uint8 frame (H×W×3)."""
        if self._proc and self._proc.stdin:
            try:
                self._proc.stdin.write(frame.tobytes())
            except BrokenPipeError:
                pass  # ffmpeg exited early — will be caught in close()

    def close(self) -> bool:
        """
        Close stdin pipe and wait for ffmpeg to finish encoding.
        Returns True on success, False if ffmpeg failed.
        """
        if self._proc is None:
            return False
        try:
            self._proc.stdin.close()
            self._proc.wait(timeout=120)   # give ffmpeg up to 2 min
            ok = self._proc.returncode == 0
            self._proc = None
            return ok
        except sp.TimeoutExpired:
            self._proc.kill()
            self._proc = None
            return False
        except Exception:
            return False


def _probe_video(path: str) -> Tuple[float, int, int, int]:
    """
    Read video metadata robustly.
    Returns (fps, width, height, reported_frame_count).
    Falls back to safe defaults if metadata is missing/corrupt.
    """
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0 or fps > 300: fps = 25.0

    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # If dimensions missing, read one frame to detect
    if W == 0 or H == 0:
        ret, probe = cap.read()
        if ret: H, W = probe.shape[:2]; cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        else: cap.release(); raise ValueError("Video has no readable frames")

    reported = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return fps, W, H, max(reported, 0)


print("✅ FfmpegWriter ready — no more stuck-at-99% hangs.")


✅ FfmpegWriter ready — no more stuck-at-99% hangs.


---
## 🔄 Section 7 — Core Processing Pipeline

All seven memory-leak and timeout bugs are fixed here.

In [ ]:
# ================================================================
# SECTION 7 — CORE VIDEO PROCESSING PIPELINE
#
# FIX SUMMARY:
#   ✅ Read loop: while True / cap.read() — no frame-count reliance
#   ✅ FfmpegWriter — no cv2.VideoWriter hang at release()
#   ✅ retina_masks=False — prevents 79MB/frame mask accumulation
#   ✅ del yolo_results each frame — immediate GPU tensor release
#   ✅ torch.cuda.empty_cache() every 10 frames
#   ✅ OOM recovery — catches torch.cuda.OutOfMemoryError
#   ✅ max_det=50 — caps detection list size
#   ✅ Progress stays < 1.0 until files fully written
# ================================================================

# How many frames between GPU cache clears
CACHE_CLEAR_INTERVAL = 10


def process_video(
    input_path    : str,
    alpha         : float = 0.55,
    yolo_conf     : float = 0.35,
    inf_size      : int   = 512,
    seg_skip      : int   = 2,
    max_frames    : int   = 0,
    preserve_persons: bool = True,
    draw_boxes    : bool  = True,
    draw_labels   : bool  = True,
    temporal_smooth:bool  = True,
    progress      = None,
) -> Tuple[str, str, Dict]:
    """
    Process every frame of the input video.
    Returns (colorized_path, segmentation_path, stats).
    """

    # ── Reset state ──────────────────────────────────────────────
    SMOOTHER.reset()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache(); gc.collect()

    # ── Open & probe video ───────────────────────────────────────
    fps, W, H, reported = _probe_video(str(input_path))
    print(f"\nVideo  : {Path(input_path).name}")
    print(f"Size   : {W}×{H}  FPS:{fps:.2f}  Reported frames:{reported}")
    print(f"Limit  : {'ALL frames' if max_frames==0 else str(max_frames)+' frames'}")

    cap = cv2.VideoCapture(str(input_path))
    if not cap.isOpened():
        raise ValueError(f"Cannot open: {input_path}")

    # ── Output paths ─────────────────────────────────────────────
    tmp_dir   = tempfile.mkdtemp(prefix="colorize_")
    col_path  = os.path.join(tmp_dir, "colorized.mp4")
    seg_path  = os.path.join(tmp_dir, "segmentation.mp4")

    writer_col = FfmpegWriter(col_path, fps, W, H)
    writer_seg = FfmpegWriter(seg_path, fps, W, H)

    # ── Main loop ────────────────────────────────────────────────
    frame_times : List[float] = []
    det_classes : Dict[str,int] = {}
    seg_mask    : Optional[np.ndarray] = None
    frame_idx   = 0
    oom_count   = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if max_frames > 0 and frame_idx >= max_frames:
            break

        t0 = time.perf_counter()

        # ── A: Segmentation (every seg_skip frames) ──────────────
        if seg_mask is None or frame_idx % seg_skip == 0:
            try:
                seg_mask = segment_frame(frame, inf_size)
            except torch.cuda.OutOfMemoryError:
                oom_count += 1
                torch.cuda.empty_cache(); gc.collect()
                # Retry at half resolution
                try:
                    seg_mask = segment_frame(frame, inf_size // 2)
                except Exception:
                    if seg_mask is None:
                        # Create blank mask as last resort
                        seg_mask = np.zeros((H,W), dtype=np.uint8)

        # ── B: YOLO detection ────────────────────────────────────
        yolo_results = None
        try:
            yolo_results = YOLO_MODEL.predict(
                source  = frame,
                conf    = yolo_conf,
                verbose = False,
                device  = DEVICE,
                retina_masks = False,   # ← CRITICAL: prevents 79MB/frame OOM
                max_det = 50,           # ← cap detection list
            )
            for r in yolo_results:
                if r.boxes is not None:
                    for cid in r.boxes.cls.cpu().numpy().astype(int):
                        n = r.names.get(cid, str(cid))
                        det_classes[n] = det_classes.get(n, 0) + 1
        except torch.cuda.OutOfMemoryError:
            oom_count += 1
            torch.cuda.empty_cache(); gc.collect()
            yolo_results = None

        # ── C: Colorize ──────────────────────────────────────────
        colorized = colorize_frame(
            bgr=frame, seg_mask=seg_mask, yolo_results=yolo_results,
            alpha=alpha, preserve_persons=preserve_persons,
            draw_boxes=draw_boxes, draw_labels=draw_labels,
            temporal_smooth=temporal_smooth,
        )

        # ── D: Segmentation visualization ────────────────────────
        sv = seg_vis(frame, seg_mask, alpha=0.70)

        # ── E: FPS overlay ───────────────────────────────────────
        elapsed = time.perf_counter() - t0
        frame_times.append(elapsed)
        cur_fps = 1.0 / elapsed if elapsed > 1e-6 else 0.0
        txt = f"Frame {frame_idx+1}  |  {cur_fps:.1f} FPS"
        for img in (colorized, sv):
            cv2.rectangle(img, (3,3), (280,30), (0,0,0), cv2.FILLED)
            cv2.putText(img, txt, (7,23), cv2.FONT_HERSHEY_SIMPLEX,
                        0.58, (0,255,0), 2, cv2.LINE_AA)

        # ── F: Write frames via ffmpeg ────────────────────────────
        writer_col.write(colorized)
        writer_seg.write(sv)

        # ── G: Mandatory per-frame cleanup ───────────────────────
        # Delete YOLO result tensors immediately — this is the
        # single most important memory fix.
        if yolo_results is not None:
            del yolo_results
        del colorized, sv

        frame_idx += 1

        # Clear GPU cache every N frames
        if DEVICE.type == "cuda" and frame_idx % CACHE_CLEAR_INTERVAL == 0:
            torch.cuda.empty_cache()

        # Progress update — stay strictly below 1.0 until done
        if progress is not None:
            denom = reported if reported > 1 else max(frame_idx + 1, 1)
            frac  = min(frame_idx / denom, 0.97)
            progress(frac, desc=f"Processing frame {frame_idx}...")

        if frame_idx % 20 == 0:
            print(f"  [{frame_idx:5d}] {cur_fps:.1f} FPS | "
                  f"{elapsed*1000:.0f} ms | OOM:{oom_count}")

    # ── Finalize ─────────────────────────────────────────────────
    cap.release()
    print(f"\n  Closing video writers (ffmpeg encoding)...")

    ok_col = writer_col.close()
    ok_seg = writer_seg.close()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache(); gc.collect()

    # Validate outputs
    for label, path, ok in [("Colorized", col_path, ok_col),
                             ("Segmentation", seg_path, ok_seg)]:
        if not os.path.exists(path):
            raise RuntimeError(f"{label} output not created: {path}")
        kb = os.path.getsize(path) / 1024
        if kb < 1:
            raise RuntimeError(
                f"{label} output is empty ({kb:.1f} KB).\n"
                f"Frames processed: {frame_idx}. ffmpeg may have failed."
            )
        print(f"  ✅ {label}: {kb:.0f} KB → {path}")

    if frame_idx == 0:
        cap.release()
        raise RuntimeError("No frames were read. File may be corrupt or unsupported.")

    # Statistics
    n   = len(frame_times)
    avg_fps = n / sum(frame_times) if frame_times else 0
    avg_ms  = sum(frame_times) / n * 1000 if n else 0
    stats = {
        "frames"     : n,
        "avg_fps"    : round(avg_fps, 2),
        "avg_ms"     : round(avg_ms, 1),
        "duration_s" : round(n / fps, 2),
        "source_fps" : round(fps, 2),
        "resolution" : f"{W}×{H}",
        "precision"  : "FP16" if USE_FP16 else "FP32",
        "oom_events" : oom_count,
        "classes"    : det_classes,
    }
    print(f"\n{'='*55}")
    print(f"  DONE: {n} frames | {avg_fps:.1f} FPS | "
          f"{stats['duration_s']}s output | OOM events: {oom_count}")
    print(f"{'='*55}")

    # Signal progress complete AFTER files are written
    if progress is not None:
        progress(1.0, desc="Done!")

    return col_path, seg_path, stats


print("✅ Pipeline ready — all 7 bugs fixed.")


✅ Pipeline ready — all 7 bugs fixed.


---
## 🖥️ Section 8 — Gradio Interface

In [ ]:
# ================================================================
# SECTION 8 — GRADIO INTERFACE
# ================================================================

GPU_BADGE = (
    f"🟢 GPU: {torch.cuda.get_device_name(0)} | FP16 Active"
    if DEVICE.type == "cuda"
    else "🟡 CPU Mode — will be slow"
)

def run_pipeline(
    video_file, alpha, yolo_conf, inf_choice,
    seg_skip, max_frames, preserve, boxes, labels, smooth,
    progress=gr.Progress(track_tqdm=True),
):
    if video_file is None:
        return None, None, "⚠️ **Please upload a video first.**"

    inf_map = {"Fast (384px)":384, "Balanced (512px)":512, "Quality (640px)":640}
    inf_sz  = inf_map.get(inf_choice, 512)
    maxf    = int(max_frames) if int(max_frames) > 0 else 0

    try:
        col, seg, stats = process_video(
            input_path     = str(video_file),
            alpha          = float(alpha),
            yolo_conf      = float(yolo_conf),
            inf_size       = inf_sz,
            seg_skip       = max(1, int(seg_skip)),
            max_frames     = maxf,
            preserve_persons= bool(preserve),
            draw_boxes     = bool(boxes),
            draw_labels    = bool(labels),
            temporal_smooth= bool(smooth),
            progress       = progress,
        )
    except Exception:
        tb = traceback.format_exc()
        return None, None, f"❌ **Error:**\n```\n{tb}\n```"

    cls_rows = "\n".join(
        f"| {k} | {v} |"
        for k,v in sorted(stats["classes"].items(), key=lambda x:-x[1])[:15]
    ) or "| — | 0 |"

    oom_warn = (
        f"\n⚠️ **{stats['oom_events']} OOM event(s) recovered automatically.**"
        if stats["oom_events"] > 0 else ""
    )

    md_out = f"""### ✅ Complete — {stats['frames']} frames processed

| Metric | Value |
|---|---|
| **Frames processed** | **{stats['frames']}** |
| Output duration | **{stats['duration_s']} s** |
| Source FPS | {stats['source_fps']} |
| Avg processing FPS | {stats['avg_fps']} |
| Avg ms / frame | {stats['avg_ms']} ms |
| Resolution | {stats['resolution']} |
| Precision | {stats['precision']} |
| OOM events | {stats['oom_events']} (auto-recovered) |

#### Detected Objects
| Class | Count |
|---|---|
{cls_rows}
{oom_warn}"""
    return col, seg, md_out


# ── Build Gradio Blocks UI ────────────────────────────────────────
with gr.Blocks(
    title="Multi-Object Colorization",
    theme=gr.themes.Soft(primary_hue="blue", secondary_hue="indigo"),
    css="""
      .hdr{background:linear-gradient(135deg,#1a1a2e,#16213e,#0f3460);
           border-radius:12px;padding:20px;margin-bottom:10px}
      .hdr h1{color:#e94560!important;margin:0;font-size:1.5em}
      .hdr p{color:#a8b2d8!important;margin:4px 0 0}
      .badge{background:#0d2137;color:#64ffda;font-family:monospace;
             border-radius:8px;padding:6px 14px;display:inline-block;margin:4px 0}
      .fix{background:#1a2e1a;color:#90ee90;border-radius:8px;
           padding:8px 14px;font-size:0.85em;margin:6px 0}
      .warn{background:#2e1a1a;color:#ee9090;border-radius:8px;
            padding:8px 14px;font-size:0.85em;margin:4px 0}
    """
) as demo:

    gr.HTML(f"""
      <div class="hdr">
        <h1>🎨 Real-Time Multi-Object Colorization</h1>
        <p>SegFormer-B2 · ADE20K 150 classes &nbsp;+&nbsp; YOLOv8s-seg · COCO 80 classes</p>
      </div>
      <div class="badge">⚡ {GPU_BADGE}</div>
      <div class="fix">
        ✅ <strong>Stable release:</strong> ffmpeg encoder · per-frame GPU cleanup ·
        OOM recovery · processes every frame of your video
      </div>
    """)

    with gr.Row():
        # ── LEFT panel ───────────────────────────────────────────
        with gr.Column(scale=1, min_width=330):
            gr.Markdown("### 📁 Upload Video")
            video_input = gr.Video(
                label="Upload video (MP4 · AVI · MOV · MKV)",
                sources=["upload"],
            )

            gr.Markdown("### ⚙️ Colorization")
            alpha = gr.Slider(0.1, 1.0, 0.25, step=0.05,
                label="Color Blend Strength (α)",
                info="0 = original  ↔  1 = full colorization")

            gr.Markdown("### 🤖 Detection")
            yolo_conf = gr.Slider(0.10, 0.90, 0.35, step=0.05,
                label="YOLO Confidence Threshold")

            gr.Markdown("### ⚡ Speed / Quality")
            inf_choice = gr.Dropdown(
                ["Fast (384px)", "Balanced (512px)", "Quality (640px)"],
                value="Balanced (512px)",
                label="Segmentation Resolution",
                info="Smaller = faster; larger = more detail")
            seg_skip = gr.Slider(1, 5, 2, step=1,
                label="SegFormer Frame Skip",
                info="2 = run segmentation every 2nd frame (faster)")
            max_frames = gr.Number(
                value=0, precision=0,
                label="Max Frames  (0 = process ALL frames)",
                info="Leave at 0 to process your entire video")

            gr.Markdown("### 🔧 Options")
            preserve = gr.Checkbox(True,  label="Preserve Person Skin Tones")
            boxes    = gr.Checkbox(True,  label="Draw Bounding Boxes")
            labels   = gr.Checkbox(True,  label="Draw Class Labels")
            smooth   = gr.Checkbox(True,  label="Temporal Smoothing (EMA)")

            gr.HTML('<div class="warn">⚠️ Keep <b>Max Frames = 0</b> '
                    'to process your complete video. '
                    'Any value > 0 hard-caps the frame count.</div>')

            btn = gr.Button("🚀 Process Complete Video",
                            variant="primary", size="lg")

        # ── RIGHT panel ──────────────────────────────────────────
        with gr.Column(scale=2):
            gr.Markdown("### 🎬 Output")
            with gr.Tabs():
                with gr.Tab("🎨 Colorized + Detection"):
                    out_col = gr.Video(
                        label="Colorized (YOLO boxes + semantic colors)",
                        interactive=False)
                with gr.Tab("🗺️ Segmentation Map"):
                    out_seg = gr.Video(
                        label="Semantic segmentation visualization",
                        interactive=False)
                with gr.Tab("📊 Statistics"):
                    out_stats = gr.Markdown("_Run the pipeline to see stats._")

    with gr.Accordion("🎨 Color Reference (ADE20K)", open=False):
        gr.Markdown("""
| Object | Color | ADE20K Class |
|---|---|---|
| 🔵 Sky | Sky Blue #87CEEB | 2 |
| 🟢 Grass | Lawn Green | 7 |
| 🌲 Tree | Forest Green #228B22 | 4 |
| ⬜ Road | Gray #808080 | 6 |
| 🟫 Building | Sandy Brown #D2B48C | 1 |
| 🔴 Car | Red #FF4444 | 17 |
| 👤 Person | **Original preserved** | 9 |
| 💧 Water / Sea / River | Dodger Blue #1E90FF | 18/21/29 |
| 🟠 Bus | Orange | YOLO |
| 🌿 Plant | Medium Green | 14 |
        """)

    with gr.Accordion("💡 Tips", open=False):
        gr.Markdown("""
- **Max Frames = 0** → processes your entire video, always
- **Balanced 512px + skip=2** → good balance of speed and quality
- **Fast 384px + skip=3** → best speed for long videos
- If you still see a connection error: re-run Sections 1-8 first
- Download via the ⬇ button inside each video player
        """)

    btn.click(
        fn=run_pipeline,
        inputs=[video_input, alpha, yolo_conf, inf_choice,
                seg_skip, max_frames, preserve, boxes, labels, smooth],
        outputs=[out_col, out_seg, out_stats],
    )

print("✅ Gradio interface ready.")


✅ Gradio interface ready.


---
## 🚀 Section 9 — Launch

In [ ]:
# ================================================================
# SECTION 9 — LAUNCH
# share=True provides public URL in Colab.
# ================================================================
print("Starting Gradio server...")
demo.launch(
    share       = True,
    show_error  = True,
    server_name = "0.0.0.0",
    quiet       = False,
)


Starting Gradio server...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://90c33519cbda7ab70a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## 🔍 Section 10 — Video Diagnostic Tool

Run this cell on any video file to see its true frame count and metadata.

In [ ]:
# ================================================================
# SECTION 10 — VIDEO DIAGNOSTIC
# Reads every frame to confirm true count vs reported count.
# ================================================================

def diagnose(path: str):
    print(f"\n{'='*55}")
    print(f"DIAGNOSTIC: {Path(path).name}")
    print(f"{'='*55}")
    if not os.path.exists(path):
        print("  ❌ File not found"); return
    print(f"  File size : {os.path.getsize(path)/1024:.1f} KB")
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        print("  ❌ OpenCV cannot open this file"); return
    fps_m = cap.get(cv2.CAP_PROP_FPS)
    fc_m  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    W_m   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H_m   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f"  Metadata  : {fc_m} frames | {fps_m:.2f} fps | {W_m}×{H_m}")
    actual = 0
    while True:
        ret, _ = cap.read()
        if not ret: break
        actual += 1
    cap.release()
    dur = actual / fps_m if fps_m > 0 else 0
    print(f"  Actual    : {actual} frames | {dur:.3f} s at {fps_m:.2f} fps")
    if fc_m == 0 or abs(fc_m - actual) > 2:
        print(f"  ⚠️  Metadata mismatch! Pipeline uses read-loop → will get all {actual} frames.")
    else:
        print(f"  ✅ Metadata OK — no truncation expected.")
    print(f"{'='*55}")


def make_test_video(path: str, n: int = 20, fps: float = 30.0):
    """Create a synthetic test video with colored bands."""
    W, H = 320, 240
    writer = FfmpegWriter(path, fps, W, H)
    for i in range(n):
        f = np.zeros((H,W,3), dtype=np.uint8)
        f[:80]  = [235,206,135]   # sky
        f[80:160] = [34,100,34]   # grass
        f[160:]   = [128,128,128] # road
        cv2.putText(f, f"Frame {i+1}/{n}", (10,50),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)
        writer.write(f)
    writer.close()
    print(f"Test video: {path}  ({n} frames @ {fps} fps)")


test_vid = "/tmp/test_input.mp4"
make_test_video(test_vid, n_frames := 20)
diagnose(test_vid)


---
## ✅ Section 11 — Full Pipeline Test (No GUI)

Verifies that all 20 frames of the test video are processed correctly.

In [ ]:
# ================================================================
# SECTION 11 — FULL PIPELINE VERIFICATION TEST
# ================================================================
import matplotlib.pyplot as plt

def run_test():
    print("="*55)
    print("FULL PIPELINE TEST")
    print("="*55)

    test_in = "/tmp/pipeline_verify.mp4"
    make_test_video(test_in, 15)

    print("\nProcessing with max_frames=0 (ALL frames)...")
    col_out, seg_out, stats = process_video(
        input_path = test_in,
        max_frames = 0,       # ALL frames
        inf_size   = 384,     # fast for test
        seg_skip   = 1,
    )

    assert stats["frames"] == 15, (
        f"❌ Expected 15 frames, got {stats['frames']}")
    print(f"\n✅ PASS: {stats['frames']}/15 frames processed")
    print(f"   Duration : {stats['duration_s']}s  FPS:{stats['avg_fps']}")
    print(f"   OOM events: {stats['oom_events']}")

    # Show sample frame
    cap = cv2.VideoCapture(col_out)
    ret, fr = cap.read(); cap.release()
    if ret:
        fig, axes = plt.subplots(1,2,figsize=(12,4))
        cap2 = cv2.VideoCapture(seg_out)
        ret2, fr2 = cap2.read(); cap2.release()
        for ax, img, ttl in zip(axes,
            [fr, fr2 if ret2 else fr],
            ["Colorized Output", "Segmentation Visualization"]):
            ax.imshow(cv2.cvtColor(img,cv2.COLOR_BGR2RGB))
            ax.set_title(ttl, fontweight="bold")
            ax.axis("off")
        plt.suptitle(f"Test PASSED — {stats['frames']} frames | "
                     f"{stats['avg_fps']} FPS | {stats['duration_s']}s",
                     fontsize=13)
        plt.tight_layout()
        plt.savefig("/tmp/test_result.png", dpi=130)
        plt.show()

    print("="*55)
    print("✅ ALL CHECKS PASSED — system is stable")
    print("="*55)

run_test()
